# 06 · The production model on the same backtest, text off

## Executive summary (read this first)

**The question.** How does the model the team submits today score on the same backtest as M0, M1, M1.1 and M1.2?

**The model:** `forecast_models.build_draws` exactly as the submission calls it, with the unit's family settings
(`forecast_models.WALK_SETTINGS`), **the text off** (shift 0, widen 1, skew 0: there is no news corpus for past dates),
500 draws, seed 0. On the four monthly units it is given the same number of monthly steps as the answer (at a past date
the panel has no publication lag, so its own month count would miscount). Production today:

| family | window | widen | shocks | sd | centre |
|---|---|---|---|---|---|
| F1 | 130 | 0.9 | Student-t ν 5 | EWMA, halflife 21 | last value, no drift |
| F2 | 130 | 1.1 | Student-t ν 4 | plain sd | last value, no drift |
| F3 | 130 | 1.2 | Student-t ν 4 | plain sd | last value, no drift |
| F4 | 260 | 1.2 | Student-t ν 5 | EWMA, halflife 63 | last value, no drift |

On log-return units production turns each row with `log1p` (it reads the rows as simple returns); M0 reads them as log
returns.

**The test:** notebook 01's backtest dates and answers; competition score per unit = `Σ weight × mean(production's category
score) ÷ mean(M0's)`. Below 1, production beats M0.

**Caution:** production's family settings were tuned (hyperparameter_tuning/01) on each unit's last 40 % of dates, which
overlaps this block, so its number here is somewhat optimistic. M0, M1, M1.1 and M1.2 tune nothing on it.

Results: §3 (per unit), §4 (per family and asset type), §5 (every model side by side), §7 (in short). No realized outcome
is read.

## 0 — Setup: notebook 01's backtest dates, answers and M0 scores

In [1]:
import multiprocessing as mp, pathlib, sys, time, tomllib, warnings
from unittest import mock
import numpy as np, pandas as pd

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0, forecast_agent as fa, forecast_models as fm
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
CARDS = {u: tomllib.loads((UNITS / u / "card.toml").read_text()) for u in UNIT_IDS}

DATES = {}
for u, g in ANSWERS.groupby("unit"):
    t = CARDS[u]["targets"]
    order = {(a, h): i for i, (a, h) in enumerate((a, int(h)) for a in t["asset_ids"] for h in t["horizons"])}
    DATES[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy(),
                 gg.drop_duplicates("horizon").set_index("horizon").steps.to_dict()) for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {len(M0_SCORES):,} backtest dates (from notebook 01)")
print("production settings:", {f: v for f, v in fm.WALK_SETTINGS.items()})

103 units, 23,635 backtest dates (from notebook 01)
production settings: {'T2-F1': {'window': 130, 'widen': 0.9, 'nu': 5, 'halflife': 21}, 'T2-F2': {'window': 130, 'widen': 1.1, 'nu': 4, 'halflife': None}, 'T2-F3': {'window': 130, 'widen': 1.2, 'nu': 4, 'halflife': None}, 'T2-F4': {'window': 260, 'widen': 1.2, 'nu': 5, 'halflife': 63}}


## 1 — Score production at every backtest date

`build_draws(panels, assets, horizons, as-of = the backtest date, text off, 500 draws, seed 0, target type, family)`: the
call the submission makes, with the as-of moved back to each backtest date.

In [2]:
NEUTRAL = {"shift": 0.0, "widen": 1.0, "skew": 0.0}

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)

def score_unit(u):
    card = CARDS[u]; t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    panels, monthly = fa._read_panels(UNITS / u), card["task"]["id"] in m0._MONTHLY_STEPS
    out = []
    for o, y, steps in DATES[u]:
        st = np.array([float(steps[h]) for h in horizons])
        with mock.patch.object(fm, "_panel_steps", (lambda s, hz, asof: st) if monthly else fm._panel_steps):
            d = fm.build_draws(panels, assets, horizons, o, {a: dict(NEUTRAL) for a in assets}, 500, 0,
                               target_type=t.get("target_type"), family=card["metadata"]["category"])
        out.append({"unit": u, "family": card["metadata"]["category"], "origin": o, **parts(card, d.reshape(500, -1), y)})
    return out

t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    PROD_SCORES = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
assert len(PROD_SCORES) == len(M0_SCORES)
print(f"scored production on {len(PROD_SCORES):,} dates in {time.time() - t0:.0f}s")

scored production on 23,635 dates in 152s


## 2 — The competition score per unit

`score = Σ w_k × mean(category_k of production) ÷ mean(category_k of M0)` over the unit's backtest dates. M0 = 1.0.

In [3]:
cats = {"marginal": "CRPS", "joint": "variogram", "tail": "tail"}
mean0 = M0_SCORES.groupby("unit")[list(cats) + ["composite"]].mean()

def vs_m0(scores):
    m = scores.groupby("unit")[list(cats) + ["composite"]].mean()
    out = {}
    for u in UNIT_IDS:
        t = CARDS[u]["targets"]; w = m0.card_weights(CARDS[u], len(t["asset_ids"]) * len(t["horizons"]))
        out[u] = sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, w) if wk)
    return pd.Series(out), m

prod_score, meanp = vs_m0(PROD_SCORES)
def asset_type(u):
    ts = sorted({RULES["assets"].get(a, "other") for a in CARDS[u]["targets"]["asset_ids"]})
    return ts[0] if len(ts) == 1 else " + ".join(ts)
rows = []
for u in UNIT_IDS:
    t = CARDS[u]["targets"]; w = m0.card_weights(CARDS[u], len(t["asset_ids"]) * len(t["horizons"]))
    r = {"family": CARDS[u]["metadata"]["category"], "unit": u, "asset type": asset_type(u),
         "dates": int((PROD_SCORES.unit == u).sum())}
    for (k, name), wk in zip(cats.items(), w):
        r[f"{name} ÷ M0"] = meanp.loc[u, k] / mean0.loc[u, k] if wk else np.nan
    r["production weighted sum"], r["M0 weighted sum"] = meanp.loc[u, "composite"], mean0.loc[u, "composite"]
    r["production vs M0"] = prod_score[u]
    rows.append(r)
RESULT = pd.DataFrame(rows).set_index(["family", "unit"])

## 3 — Per unit

In [4]:
RESULT.sort_values("production vs M0")

asset type  dates  CRPS ÷ M0  variogram ÷ M0  tail ÷ M0  production weighted sum  M0 weighted sum  production vs M0
family unit                                                                                                                                                     
T2-F1  t2-F1-measured-pace-2004                   rates     51      0.424          0.7125     0.6297                    0.178           0.3798            0.5517
       t2-F1-conundrum-2005                       rates     60     0.5363          0.7504     0.6613                   0.1711            0.301            0.6256
T2-F2  t2-F2-considerable-removal-2004            rates     52     0.6806             NaN     0.8442                   0.1439            0.209            0.7274
T2-F1  t2-F1-ai-mom-2024                  equity_factor    355     0.7271             NaN     0.8772                  0.03801          0.05185              0.77
       t2-F1-liftoff-telegraph-2015               rates    221     0.8536          0.6127      0.865                  0.06217          0.07852            0.7836
       t2-F1-patient-dropped-2015                 rates    215      0.791             NaN     0.7734                  0.06975          0.08828             0.786
       t2-F1-chf-highly-valued-2021                  fx    306     0.7851          0.8461     0.8009                  0.01417          0.01777            0.8065
T2-F4  t2-F4-funding-stress-2y-2008               rates    121     0.8765             NaN     0.6372                    0.284           0.3293            0.8081
T2-F1  t2-F1-dkk-peg-2019                            fx    275     0.7661             NaN     0.9536                   0.1933           0.2487            0.8197
       t2-F1-normalization-2017                   rates    249       0.74             NaN      1.041                   0.2082           0.2773             0.826
T2-F3  t2-F3-taper-steepener-2013                 rates    188     0.8074          0.7806     0.9496                   0.6601           0.8382            0.8278
T2-F4  t2-F4-hike-cycle-2021Q4b                   rates    317      0.866             NaN     0.7361                   0.2365           0.2755            0.8289
T2-F1  t2-F1-eur-range-2017                          fx    250     0.6887           0.998     0.9269                  0.03139          0.04264            0.8291
T2-F2  t2-F2-taper-begins-2013                    rates    197     0.8248             NaN     0.8526                   0.3087           0.3738            0.8327
T2-F3  t2-F3-dots-vs-oil-2014                     rates    212     0.8251          0.7394     0.9932                   0.3755           0.4895             0.833
T2-F2  t2-F2-eur-parity-2022                         fx    329     0.8382             NaN     0.8297                  0.01626          0.01941            0.8358
T2-F3  t2-F3-conundrum-joint-2005                 rates     64     0.7476          0.9784      0.851                   0.9493            1.021            0.8375
T2-F2  t2-F2-ecb-qe-telegraph-2014                   fx    216     0.8043             NaN     0.9272                   0.0239          0.02954            0.8394
       t2-F2-draghi-precommit-2014                   fx    208     0.7984             NaN     0.9491                  0.02528          0.03145            0.8414
T2-F1  t2-F1-qe3-end-2014                         rates    206     0.7662          0.8861     0.9739                   0.2579           0.3274            0.8437
T2-F2  t2-F2-whatever-it-takes-2012                  fx    180     0.8133             NaN     0.9589                  0.03269          0.03993            0.8549
       t2-F2-aud-taper-2013                          fx    193     0.8272             NaN     0.9259                  0.02188          0.02632            0.8554
       t2-F2-trade-war-escalation-2019            rates    285     0.8521             NaN     0.9144                   0.1307           0.1528            0.8699
T2-F3  t2-F3-rates-i

## 4 — Per family and per asset type

In [5]:
def summary(g):
    return {"units": len(g), "production vs M0 (mean)": g["production vs M0"].mean(), "median": g["production vs M0"].median(),
            "better than M0": f'{(g["production vs M0"] < 1).sum()}/{len(g)}', "past 1.20": int((g["production vs M0"] > 1.2).sum()),
            "CRPS ÷ M0 (geo)": geo(g["CRPS ÷ M0"]), "tail ÷ M0 (geo)": geo(g["tail ÷ M0"])}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*RESULT.groupby("family"), ("all", RESULT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in RESULT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)

per family:


,units,production vs M0 (mean),median,better than M0,past 1.20,CRPS ÷ M0 (geo),tail ÷ M0 (geo)
family,,,,,,,
T2-F1,23,0.9288,0.8929,16/23,2,0.8556,0.9645
T2-F2,27,1.011,0.9579,16/27,3,0.9674,1.045
T2-F3,22,0.9274,0.9298,21/22,0,0.9155,0.9453
T2-F4,31,1.003,0.9858,21/31,1,1.007,0.9661
all,103,0.9726,0.9509,74/103,6,0.9414,0.9813


per asset type:


,units,production vs M0 (mean),median,better than M0,past 1.20,CRPS ÷ M0 (geo),tail ÷ M0 (geo)
asset type,,,,,,,
cpi,2,1.308,1.308,0/2,1,1.352,1.264
equity_factor,16,0.9935,0.9995,8/16,0,0.9863,0.9861
fx,35,0.9819,0.9579,27/35,2,0.9378,1.023
fx + rates,5,0.9429,0.9365,5/5,0,0.9451,0.9307
payrolls,1,1.624,1.624,0/1,1,1.815,1.144
rates,43,0.9285,0.919,34/43,2,0.8973,0.9403
unemployment,1,1.036,1.036,0/1,0,0.9889,0.9104


## 5 — Every model so far, side by side

The mean competition score vs M0 over units, from each notebook's saved scores (same dates, same answers). M1.2 is
notebook 05's: the Student-t fitted on each window (3 months to 2 years), M0's drift kept.

In [6]:
_w12 = pd.read_parquet(DATA / "m1_2_window_backtest_scores.parquet")
models = {"M0": M0_SCORES, "M1 (log returns)": pd.read_parquet(DATA / "m1_backtest_scores.parquet"),
          "M1.1 (per asset type)": pd.read_parquet(DATA / "m1_1_backtest_scores.parquet"),
          **{f"M1.2 (t fitted, {lab})": _w12[_w12.window == W].drop(columns="window")
             for W, lab in [(63, "3 months"), (126, "6 months"), (252, "1 year"), (300, "300 rows"), (504, "2 years")]},
          "production (text off)": PROD_SCORES}
fam = pd.Series({u: CARDS[u]["metadata"]["category"] for u in UNIT_IDS})
LADDER = pd.DataFrame({name: {**{f: vs_m0(sc)[0][fam[fam == f].index].mean() for f in sorted(fam.unique())},
                              "all": vs_m0(sc)[0].mean()} for name, sc in models.items()}).T
LADDER["better than M0"] = [f"{(vs_m0(sc)[0] < 1).sum()}/{len(UNIT_IDS)}" for sc in models.values()]
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1 (log returns),1.324,1.096,1.034,1.016,1.11,36/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
"M1.2 (t fitted, 3 months)",1.517,1.205,1.37,1.097,1.277,8/103
"M1.2 (t fitted, 6 months)",1.162,1.116,1.143,1.036,1.108,19/103
"M1.2 (t fitted, 1 year)",1.008,1.017,1.004,0.998,1.006,45/103
"M1.2 (t fitted, 300 rows)",0.9911,0.9995,0.9982,0.993,0.9954,62/103
"M1.2 (t fitted, 2 years)",0.97,0.9967,0.9761,1.01,0.9902,64/103
production (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103


## 6 — Save

In [7]:
PROD_SCORES.to_parquet(DATA / "production_backtest_scores.parquet", index=False)
RESULT.to_csv(DATA / "production_vs_m0_per_unit.csv")
FAMILY.to_csv(DATA / "production_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "production_vs_m0_by_asset_type.csv")
LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for f in ("production_backtest_scores.parquet", "production_vs_m0_per_unit.csv", "production_vs_m0_per_family.csv",
          "production_vs_m0_by_asset_type.csv", "all_models_vs_m0.csv"):
    print(f"data/{f:<36} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/production_backtest_scores.parquet      842.4 kB
data/production_vs_m0_per_unit.csv            15.2 kB
data/production_vs_m0_per_family.csv           0.6 kB
data/production_vs_m0_by_asset_type.csv        0.8 kB
data/all_models_vs_m0.csv                      1.1 kB


## 7 — In short

In [8]:
a = FAMILY.loc["all"]
print(f"Production, text off, on the same {len(PROD_SCORES):,} backtest dates: {a['production vs M0 (mean)']:.4f} vs M0 "
      f"(better on {a['better than M0']}, {int(a['past 1.20'])} units past 1.20).")
for f in [f for f in FAMILY.index if f != "all"]:
    r = FAMILY.loc[f]
    print(f"  {f}: {r['production vs M0 (mean)']:.4f}  ({r['better than M0']} better)")
print("every model, mean over units: " + "; ".join(f"{n} {LADDER.loc[n, 'all']:.4f}" for n in LADDER.index))
print("Development submission 01 (one real date per unit, text on): 0.9528.")

Production, text off, on the same 23,635 backtest dates: 0.9726 vs M0 (better on 74/103, 6 units past 1.20).
  T2-F1: 0.9288  (16/23 better)
  T2-F2: 1.0114  (16/27 better)
  T2-F3: 0.9274  (21/22 better)
  T2-F4: 1.0033  (21/31 better)
every model, mean over units: M0 1.0000; M1 (log returns) 1.1095; M1.1 (per asset type) 0.9976; M1.2 (t fitted, 3 months) 1.2775; M1.2 (t fitted, 6 months) 1.1081; M1.2 (t fitted, 1 year) 1.0065; M1.2 (t fitted, 300 rows) 0.9954; M1.2 (t fitted, 2 years) 0.9902; production (text off) 0.9726
Development submission 01 (one real date per unit, text on): 0.9528.
